In [ ]:
# Set the data folder before loading the CSV files.
from pathlib import Path
SEED=42
DATA_DIR=Path("data")

In [15]:
# pandas reads the CSV files, and display shows tables in the notebook.
import pandas as pd
from IPython.display import display

intent_file=DATA_DIR / "intent_dataset.csv"
amb_file=DATA_DIR / "ambiguity_dataset.csv"

intent_df=pd.read_csv(intent_file)
amb_df=pd.read_csv(amb_file)

intent_cols={"question", "intent", "split", "data_origin"}
amb_cols={"question", "ambiguity", "split", "data_origin"}
assert intent_cols.issubset(intent_df.columns)
assert amb_cols.issubset(amb_df.columns)
assert set(intent_df["split"].unique()) == {"train", "test"}
assert set(amb_df["split"].unique()) == {"train", "test"}

print("Intent dataset:", len(intent_df), "rows")
print("Ambiguity dataset:", len(amb_df), "rows")
print("Intent examples in each split:")
display(intent_df.groupby(["split", "intent"]).size().unstack(fill_value=0))
print("Ambiguity examples in each split:")
display(amb_df.groupby(["split", "ambiguity"]).size().unstack(fill_value=0))
print("Intent data origins:", sorted(intent_df["data_origin"].unique()))
print("Ambiguity data origins:", sorted(amb_df["data_origin"].unique()))

Intent dataset: 8500 rows
Ambiguity dataset: 4250 rows
Intent examples in each split:


intent,AVERAGE_SALARY,COUNT_EMPLOYEES,DELETE_EMPLOYEE,FILTER_BY_AGE,FILTER_BY_DEPARTMENT,FILTER_BY_SALARY,GROUP_BY_DEPARTMENT,INSERT_EMPLOYEE,MAXIMUM_SALARY,MINIMUM_SALARY,ORDER_BY_SALARY_ASC,ORDER_BY_SALARY_DESC,SELECT_ALL,UPDATE_AGE,UPDATE_DEPARTMENT,UPDATE_NAME,UPDATE_SALARY
split,,,,,,,,,,,,,,,,,
test,100,100,100,100,100,100,100,100,100,100,100,100,100,100,100,100,100
train,400,400,400,400,400,400,400,400,400,400,400,400,400,400,400,400,400


Ambiguity examples in each split:


ambiguity,CLEAR,NEEDS_CLARIFICATION
split,,
test,425,425
train,1700,1700


Intent data origins: ['synthetic_template_augmented']
Ambiguity data origins: ['synthetic_template_augmented']


In [16]:
# re cleans whitespace; TF-IDF turns text into numeric features.
import re
from sklearn.feature_extraction.text import TfidfVectorizer


def clean_text(text):
    """Normalize a question without deleting meaning-changing words."""
    text=text.strip().lower()
    return re.sub(r"\s+", " ", text)


def make_tfidf():
    """Create the same TF-IDF feature pipeline for training and prediction."""
    return TfidfVectorizer(
        preprocessor=clean_text,
        lowercase=False,
        ngram_range=(1, 2),
        min_df=1,
    )


example="  NOT   below 50000 "
print("Original:", repr(example))
print("Preprocessed:", clean_text(example))

Original: '  NOT   below 50000 '
Preprocessed: not below 50000


In [17]:
# Import evaluation tools only when we reach the evaluation section.
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)


def show_scores(name, model, vec, questions, answers):
    """Print held-out scores; these are evaluation results, not SQL scores."""
    x=vec.transform(questions)
    pred=model.predict(x)
    labels=sorted(set(answers) | set(pred))

    precision, recall, f1, _=precision_recall_fscore_support(
        answers,
        pred,
        labels=labels,
        average="macro",
        zero_division=0,
    )

    print(name)
    print("Accuracy:", round(accuracy_score(answers, pred), 3))
    print("Macro precision:", round(precision, 3))
    print("Macro recall:", round(recall, 3))
    print("Macro F1:", round(f1, 3))
    print("\nClassification report:")
    print(classification_report(answers, pred, zero_division=0))
    print("Confusion matrix; label order:", labels)
    display(pd.DataFrame(
        confusion_matrix(answers, pred, labels=labels),
        index=["actual " + label for label in labels],
        columns=["predicted " + label for label in labels],
    ))
    return pred

In [18]:
# The ANN model is imported here because this is where it is used.
from sklearn.neural_network import MLPClassifier

int_train=intent_df[intent_df["split"] == "train"]
int_test=intent_df[intent_df["split"] == "test"]

int_vec=make_tfidf()
X_train=int_vec.fit_transform(int_train["question"])
X_test=int_vec.transform(int_test["question"])
y_train=int_train["intent"]
y_test=int_test["intent"]

ann = MLPClassifier(
    hidden_layer_sizes=(32,),
    activation="relu",
    solver="lbfgs",
    alpha=0.01,
    max_iter=500,
    random_state=SEED,
)
ann.fit(X_train, y_train)

print("ANN training rows:", len(int_train))
print("ANN held-out test rows:", len(int_test))
ann_pred=show_scores(
    "ANN / MLPClassifier — intent classification",
    ann,
    int_vec,
    int_test["question"],
    y_test,
)

ANN training rows: 6800
ANN held-out test rows: 1700
ANN / MLPClassifier — intent classification
Accuracy: 0.829
Macro precision: 0.866
Macro recall: 0.829
Macro F1: 0.805

Classification report:
                      precision    recall  f1-score   support

      AVERAGE_SALARY       1.00      1.00      1.00       100
     COUNT_EMPLOYEES       0.62      1.00      0.77       100
     DELETE_EMPLOYEE       0.64      1.00      0.78       100
       FILTER_BY_AGE       1.00      1.00      1.00       100
FILTER_BY_DEPARTMENT       0.91      1.00      0.95       100
    FILTER_BY_SALARY       0.77      0.91      0.83       100
 GROUP_BY_DEPARTMENT       1.00      0.49      0.66       100
     INSERT_EMPLOYEE       0.88      1.00      0.93       100
      MAXIMUM_SALARY       1.00      1.00      1.00       100
      MINIMUM_SALARY       1.00      1.00      1.00       100
 ORDER_BY_SALARY_ASC       0.65      0.30      0.41       100
ORDER_BY_SALARY_DESC       1.00      0.48      0.65       1

,predicted AVERAGE_SALARY,predicted COUNT_EMPLOYEES,predicted DELETE_EMPLOYEE,predicted FILTER_BY_AGE,predicted FILTER_BY_DEPARTMENT,predicted FILTER_BY_SALARY,predicted GROUP_BY_DEPARTMENT,predicted INSERT_EMPLOYEE,predicted MAXIMUM_SALARY,predicted MINIMUM_SALARY,predicted ORDER_BY_SALARY_ASC,predicted ORDER_BY_SALARY_DESC,predicted SELECT_ALL,predicted UPDATE_AGE,predicted UPDATE_DEPARTMENT,predicted UPDATE_NAME,predicted UPDATE_SALARY
actual AVERAGE_SALARY,100,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
actual COUNT_EMPLOYEES,0,100,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
actual DELETE_EMPLOYEE,0,0,100,0,0,0,0,0,0,0,0,0,0,0,0,0,0
actual FILTER_BY_AGE,0,0,0,100,0,0,0,0,0,0,0,0,0,0,0,0,0
actual FILTER_BY_DEPARTMENT,0,0,0,0,100,0,0,0,0,0,0,0,0,0,0,0,0
actual FILTER_BY_SALARY,0,9,0,0,0,91,0,0,0,0,0,0,0,0,0,0,0
actual GROUP_BY_DEPARTMENT,0,51,0,0,0,0,49,0,0,0,0,0,0,0,0,0,0
actual INSERT_EMPLOYEE,0,0,0,0,0,0,0,100,0,0,0,0,0,0,0,0,0
actual MAXIMUM_SALARY,0,0,0,0,0,0,0,0,100,0,0,0,0,0,0,0,0
actual MINIMUM_SALARY,0,0,0,0,0,0,0,0,0,100,0,0,0,0,0,0,0


In [19]:
# Random Forest is imported in its own training section.
from sklearn.ensemble import RandomForestClassifier

amb_train=amb_df[amb_df["split"] == "train"]
amb_test=amb_df[amb_df["split"] == "test"]

amb_vec=make_tfidf()
X_train=amb_vec.fit_transform(amb_train["question"])
X_test=amb_vec.transform(amb_test["question"])
y_train=amb_train["ambiguity"]
y_test=amb_test["ambiguity"]

rf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=SEED,
)
rf.fit(X_train, y_train)

print("Random Forest training rows:", len(amb_train))
print("Random Forest held-out test rows:", len(amb_test))
rf_pred=show_scores(
    "Random Forest — ambiguity classification",
    rf,
    amb_vec,
    amb_test["question"],
    y_test,
)

Random Forest training rows: 3400
Random Forest held-out test rows: 850
Random Forest — ambiguity classification
Accuracy: 0.841
Macro precision: 0.847
Macro recall: 0.841
Macro F1: 0.841

Classification report:
                     precision    recall  f1-score   support

              CLEAR       0.89      0.78      0.83       425
NEEDS_CLARIFICATION       0.80      0.91      0.85       425

           accuracy                           0.84       850
          macro avg       0.85      0.84      0.84       850
       weighted avg       0.85      0.84      0.84       850

Confusion matrix; label order: ['CLEAR', 'NEEDS_CLARIFICATION']


,predicted CLEAR,predicted NEEDS_CLARIFICATION
actual CLEAR,330,95
actual NEEDS_CLARIFICATION,40,385


In [20]:
questions=[
    "How many employees are there?",
    "Show employees in the IT department",
    "Find employees earning more than 50000",
    "Show employees with high salary",
]

X_intent=int_vec.transform(questions)
X_amb=amb_vec.transform(questions)

predictions=pd.DataFrame({
    "question": questions,
    "ANN intent": ann.predict(X_intent),
    "Random Forest ambiguity": rf.predict(X_amb),
})
display(predictions)

,question,ANN intent,Random Forest ambiguity
0,How many employees are there?,COUNT_EMPLOYEES,NEEDS_CLARIFICATION
1,Show employees in the IT department,FILTER_BY_DEPARTMENT,NEEDS_CLARIFICATION
2,Find employees earning more than 50000,FILTER_BY_SALARY,NEEDS_CLARIFICATION
3,Show employees with high salary,FILTER_BY_SALARY,NEEDS_CLARIFICATION
